# AI Playwright Test Generator — Pipeline Debugger

Run cells **top to bottom** for a full pipeline trace.  
Re-run any individual cell after editing config or patching a module — state is preserved between cells.

| Cell | Stage | What to look for |
|------|-------|------------------|
| 1 | Setup | No import errors, kernel shows `ai-playwright` |
| 2 | Config | Correct URL and story before proceeding |
| 3 | LLM Skeleton | Raw `{{CLICK:...}}` tokens present, no guessed selectors |
| 4 | Scraper | Candidate count per page, element types found |
| 5 | Resolution | Per-placeholder scores, resolved vs skipped |
| 6 | Final Output | Clean test file, skip count acceptable |

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
# Adds src/ to path and patches the event loop so sync Playwright
# doesn't clash with Jupyter's own asyncio loop.

import sys
from pathlib import Path

# Resolve project root (one level up from /notebooks)
PROJECT_ROOT = Path(globals()["_dh"][0]).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Patch asyncio BEFORE importing anything that touches Playwright
import nest_asyncio

nest_asyncio.apply()

import pandas as pd
from IPython.display import display

from src.llm_client import LLMClient  # protected — read only
from src.placeholder_orchestrator import PlaceholderOrchestrator
from src.placeholder_resolver import PlaceholderResolver
from src.semantic_candidate_ranker import SemanticCandidateRanker
from src.stateful_scraper import StatefulScraper

# Project modules
from src.test_generator import generate_test  # protected — read only

print(f"✅ Project root: {PROJECT_ROOT}")
print(f"✅ Python: {sys.version.split()[0]}")
print("✅ All imports OK")

In [ ]:
# ── Cell 2: Config ────────────────────────────────────────────────────────────
# Edit these values before running the rest of the notebook.
# This is the only cell you should need to change between debug sessions.

USER_STORY = """
As a user I want to log in with valid credentials
so that I can access my dashboard.

Acceptance criteria:
- Navigate to /login
- Enter username and password
- Click the login button
- Assert the dashboard heading is visible
"""

TARGET_URL = "http://localhost:3000"  # change to your AUT

# Model — use the dense model; avoid MoE for code gen quality
MODEL_NAME = "qwen2.5-coder:27b"
PARSE_MODE = "smart"  # smart | llm | regex
HEADLESS = True  # set False to watch the browser

print(f"Story length  : {len(USER_STORY.split())} words")
print(f"Target URL    : {TARGET_URL}")
print(f"Model         : {MODEL_NAME}")
print(f"Parse mode    : {PARSE_MODE}")
print(f"Headless      : {HEADLESS}")

In [ ]:
# ── Cell 3: LLM Skeleton Generation ───────────────────────────────────────────
# What to look for:
#   ✅ Every action step has a {{TOKEN:description}} — no raw CSS or XPath
#   ❌ If you see page.locator('#some-id') the LLM guessed — check your prompt

client = LLMClient(model=MODEL_NAME)
skeleton = generate_test(user_story=USER_STORY, llm_client=client, parse_mode=PARSE_MODE)

# ── Token inventory
import re

TOKEN_RE = re.compile(r"\{\{(\w+):([^}]+)\}\}")
tokens = TOKEN_RE.findall(skeleton)

print(f"Tokens found: {len(tokens)}")
print("-" * 60)
for kind, desc in tokens:
    print(f"  [{kind:10s}] {desc}")

print("\n" + "=" * 60 + "\nRAW SKELETON\n" + "=" * 60)
print(skeleton)

In [ ]:
# ── Cell 4: Scraper Run ───────────────────────────────────────────────────────
# What to look for:
#   ✅ Candidates > 0 for the pages your tokens reference
#   ❌ Empty candidate list usually means auth wall or wrong URL
#   ❌ Only <body> / <html> elements = JS didn't render; try headless=False

scraper = StatefulScraper(headless=HEADLESS)
candidates = scraper.scrape(TARGET_URL)

# Build a readable DataFrame
rows = []
for elem in candidates:
    rows.append(
        {
            "tag": elem.get("tag", ""),
            "text": (elem.get("text") or "")[:60],
            "role": elem.get("role", ""),
            "id": elem.get("id", ""),
            "name": elem.get("name", ""),
            "placeholder": elem.get("placeholder", ""),
            "css_selector": (elem.get("css_selector") or "")[:80],
        }
    )

df = pd.DataFrame(rows)
print(f"Total candidates scraped: {len(df)}")
print(f"Tag breakdown:\n{df['tag'].value_counts().to_string()}")
print()
display(df)

In [ ]:
# ── Cell 4b: Candidate search (optional) ──────────────────────────────────────
# Quickly filter the candidate table to find elements relevant to a specific token.
# Change SEARCH_TERM to the description from one of your failing tokens.

SEARCH_TERM = "login button"  # ← edit this

mask = df.apply(lambda row: row.astype(str).str.contains(SEARCH_TERM, case=False, na=False).any(), axis=1)

results = df[mask]
print(f"Candidates matching '{SEARCH_TERM}': {len(results)}")
display(results)

In [ ]:
# ── Cell 5: Placeholder Resolution ────────────────────────────────────────────
# This is where most bugs surface. For each token you'll see:
#   - All candidates the ranker considered
#   - Their similarity scores
#   - What won (or why it fell through to pytest.skip)
#
# Known bug watchpoints:
#   B-??? double-ranking: if top score appears twice, find_best_match fired twice
#   B-??? semantic ranker not firing: all scores identical → equality check bug

ranker = SemanticCandidateRanker()
resolver = PlaceholderResolver(candidates=candidates, ranker=ranker)
orchestrator = PlaceholderOrchestrator(resolver=resolver)

resolved_skeleton, resolution_report = orchestrator.resolve(skeleton)

# ── Per-token resolution table
report_rows = []
for token_desc, result in resolution_report.items():
    report_rows.append(
        {
            "token": token_desc[:50],
            "status": result.get("status", "unknown"),
            "winner": (result.get("selector") or "")[:60],
            "winner_score": result.get("score", None),
            "candidates": result.get("candidate_count", 0),
            "fallback": result.get("fallback", False),
        }
    )

report_df = pd.DataFrame(report_rows)
resolved = report_df[report_df["status"] == "resolved"]
skipped = report_df[report_df["status"] != "resolved"]

print(f"Resolved : {len(resolved)} / {len(report_df)}")
print(f"Skipped  : {len(skipped)} / {len(report_df)}")
print()
display(
    report_df.style.applymap(
        lambda v: "background-color: #d4edda"
        if v == "resolved"
        else "background-color: #f8d7da"
        if v != "resolved" and isinstance(v, str)
        else "",
        subset=["status"],
    )
)

In [ ]:
# ── Cell 5b: Deep-dive a single failing token ─────────────────────────────────
# Paste the token description from the skipped rows above.
# Shows every candidate the ranker saw and its raw score.

FAILING_TOKEN = "login button"  # ← paste failing token description here

scored = ranker.score_all(query=FAILING_TOKEN, candidates=candidates)

scored_df = pd.DataFrame(
    [
        {
            "score": round(s["score"], 4),
            "tag": s["element"].get("tag", ""),
            "text": (s["element"].get("text") or "")[:60],
            "role": s["element"].get("role", ""),
            "css_selector": (s["element"].get("css_selector") or "")[:80],
        }
        for s in scored
    ]
).sort_values("score", ascending=False)

print(f"Top 10 candidates for '{FAILING_TOKEN}':")
display(scored_df.head(10))

In [ ]:
# ── Cell 6: Final Output ──────────────────────────────────────────────────────
# Shows the resolved test file. Unresolved tokens become pytest.skip() calls.
# Check the skip count — if it's higher than cell 5 suggested, something
# is double-skipping downstream.

skip_count = resolved_skeleton.count("pytest.skip")
print(f"pytest.skip() calls in output: {skip_count}")
print("=" * 60 + "\nFINAL TEST FILE\n" + "=" * 60)
print(resolved_skeleton)

# Optionally write to disk for inspection
OUT_PATH = PROJECT_ROOT / "notebooks" / "_debug_output.py"
OUT_PATH.write_text(resolved_skeleton)
print(f"\nWritten to: {OUT_PATH}")

In [ ]:
# ── Cell 7: Patch and re-test a selector manually ────────────────────────────
# When cell 5b shows the right element exists but didn't win,
# use this cell to inject a manual override and re-run resolution
# without restarting the whole pipeline.
#
# This does NOT write back to the codebase — it's a scratchpad.

OVERRIDE_TOKEN = "login button"  # token description to override
OVERRIDE_SELECTOR = "button[type='submit']"  # selector you want to force

patched = resolved_skeleton.replace(
    f'pytest.skip("Unresolved: {OVERRIDE_TOKEN}")', f'page.locator("{OVERRIDE_SELECTOR}")  # manually patched'
)

patch_count = resolved_skeleton.count(f"Unresolved: {OVERRIDE_TOKEN}")
print(f"Occurrences patched: {patch_count}")
print()
print(patched)